<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 6 — Machine Learning

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif

**Tâche 1 — Régression :** prédire `duree_totale_h` (durée de traitement en heures)  
**Tâche 2 — Classification :** prédire `Cause.intervention` (type de sinistre)

**Référence :** [`docs/phase6_machine_learning.md`](../docs/phase6_machine_learning.md)  
**Données :** `data/dataset_complet.csv` — 98 935 × 34 colonnes


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, f1_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine du projet."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
dataset = pd.read_csv('data/dataset_complet.csv', encoding='utf-8')
print(f" Dataset : {dataset.shape[0]:,} lignes × {dataset.shape[1]} colonnes")

---
## Section 1 — Préparation des Données

### 1.1 Définition des features et de la cible

In [ ]:
# --- Features communes aux deux tâches
FEATURES_NUMERIQUES = [
    'agent_experience_j', 'agent_duree_travail_j', 'agent_temps_travail_pct',
    'delai_survenance_ouverture_j', 'nb_interventions', 'nb_agents_distincts',
    'mois_ouverture'
]

FEATURES_CATEGORIELLES = [
    'agent_lieu_travail', 'agent_contrat', 'agent_population', 'annee_ouverture'
]

TOUTES_FEATURES = FEATURES_NUMERIQUES + FEATURES_CATEGORIELLES

print("Features numériques  :", FEATURES_NUMERIQUES)
print("Features catégorielles:", FEATURES_CATEGORIELLES)
print(f"Total features       : {len(TOUTES_FEATURES)}")

In [ ]:
# --- Pipeline de prétraitement (M2 — évite le data leakage)
preprocesseur = ColumnTransformer(transformers=[
    ('numerique',    Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), FEATURES_NUMERIQUES),
    ('categoriel',   Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), FEATURES_CATEGORIELLES),
], remainder='drop')

print(" Pipeline de prétraitement défini")
print("   → Imputation médiane pour les numériques")
print("   → Imputation mode pour les catégoriels")
print("   → StandardScaler sur les numériques")
print("   → OneHotEncoder sur les catégoriels")

---
## Section 2 — Tâche 1 : Régression (`duree_totale_h`)

### Décision M3 : conserver les NaN via imputation (pas de dropna)
> Le `SimpleImputer` dans le pipeline gère les NaN — aucune observation n'est perdue.